<a href="https://colab.research.google.com/github/NiatKahsay/bnpl-risk-intelligence/blob/niat-branch/BNPL_Notebook1_DataCollection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 💳 BNPL Capstone -  Data Collection
**Authors:** Niyat Kahsay · Saloni Bahte · Kiara Paz | March 2026

This notebook pulls:
1. **CFPB Complaint Database** — real complaints about Affirm, Klarna, Afterpay, Zip, Sezzle
2. **FRED Economic Data** — macro indicators (delinquency rate, savings rate, etc.)
3. **Lending Club** — labeled loan default data for ML modeling


## Mount Drive & Install Packages

In [11]:
from google.colab import files
import io
from google.colab import drive
drive.mount('/content/drive')

import os
BASE = '/content/drive/MyDrive/BNPL_Capstone'
for folder in ['data/raw','data/processed','output/figures','output/models']:
    os.makedirs(f'{BASE}/{folder}', exist_ok=True)
print('Project folders ready:', BASE)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project folders ready: /content/drive/MyDrive/BNPL_Capstone


In [ ]:
!pip install -q fredapi tqdm pyarrow
print('extra packages installed')

## 1 · CFPB Consumer Complaint Database

The government has a website where people go to complain when a company treats them badly. It is called [Consumer Complaint Database](https://www.consumerfinance.gov/data-research/consumer-complaints/#get-the-data). This files downloads every complaint people filed against Affirm, Klarna, Afterpay, Zip, Sezzle, and PayPal.

This dataset has  collection of things like:



*   When did they complain?
*   What state do they live in?
*   What was the problem?
* Did the company respond on time?


This tells : where are people most angry and why?

In [5]:
import pandas as pd

# load downloaded CSV
complaints = pd.read_csv('/content/complaints.csv', low_memory=False)

# rename columns to match what the rest of the notebook expects
complaints.columns = complaints.columns.str.strip().str.lower().str.replace(' ', '_')

# filter to only BNPL companies
BNPL_COMPANIES = ['Affirm', 'Klarna', 'Afterpay', 'Zip', 'Sezzle', 'PayPal']
complaints = complaints[complaints['company'].isin(BNPL_COMPANIES)]

# Parse dates
complaints['date_received'] = pd.to_datetime(complaints['date_received'], errors='coerce')
complaints['year']  = complaints['date_received'].dt.year
complaints['month'] = complaints['date_received'].dt.month

# save clean version
complaints.to_csv(f'{BASE}/data/raw/cfpb_complaints.csv', index=False)
print(f' Saved {len(complaints):,} total complaints to Drive')
complaints.head()

✅ Saved 0 total complaints to Drive


,date_received,product,sub-product,issue,sub-issue,consumer_complaint_narrative,company_public_response,company,state,zip_code,tags,consumer_consent_provided?,submitted_via,date_sent_to_company,company_response_to_consumer,timely_response?,consumer_disputed?,complaint_id,year,month


## 2 · FRED Macro Indicators
> Get an **API** at [Federal Reserve of ST.Louis](https://fred.stlouisfed.org/docs/api/fred/v2/index.html)

FRED (Government economic data)
The Federal Reserve (the US government's money people) tracks giant numbers about the whole country's finances. This dataset has a colleciton of things like:

* How many Americans are late on their credit cards right now?
* How much money are people saving?
* How high are interest rates?
* How many people are unemployed?

This tells: when the economy is bad, do BNPL problems get worse?

In [10]:
from fredapi import Fred

FRED_API_KEY = 'f0073dd2a90a7405dea3f88d092ff267'

fred = Fred(api_key=FRED_API_KEY)

FRED_SERIES = {
        'DRCCLACBS':      'Credit Card Delinquency Rate',
        'DRSFRMACBS':     'Consumer Loan Delinquency Rate',
        'PSAVERT':        'Personal Savings Rate',
        'CCLACBW027SBOG': 'Consumer Credit: Revolving',
        'TERMCBCCALLNS':  'Credit Card Interest Rate',
        'UMCSENT':        'Consumer Sentiment Index',
        'UNRATE':         'Unemployment Rate',
        'CPIAUCSL':       'CPI (Inflation Proxy)',
    }
frames = []
for sid, name in tqdm(FRED_SERIES.items(), desc='Fetching FRED'):
        try:
            s = fred.get_series(sid, observation_start='2018-01-01')
            df = s.reset_index()
            df.columns = ['date', 'value']
            df['indicator'] = name
            df['series_id'] = sid
            frames.append(df)
            time.sleep(0.2)
        except Exception as e:
            print(f'  {sid}: {e}')
macro = pd.concat(frames, ignore_index=True)
macro.to_csv(f'{BASE}/data/raw/fred_macro.csv', index=False)
print(f'Saved {len(macro):,} rows of macro data')
macro.pivot(index='date', columns='indicator', values='value').tail()

Fetching FRED:   0%|          | 0/8 [00:00<?, ?it/s]

Saved 971 rows of macro data


indicator,CPI (Inflation Proxy),Consumer Credit: Revolving,Consumer Loan Delinquency Rate,Consumer Sentiment Index,Credit Card Delinquency Rate,Credit Card Interest Rate,Personal Savings Rate,Unemployment Rate
date,,,,,,,,
2026-01-21,NaN,1070.1461,NaN,NaN,NaN,NaN,NaN,NaN
2026-01-28,NaN,1067.5885,NaN,NaN,NaN,NaN,NaN,NaN
2026-02-04,NaN,1069.7639,NaN,NaN,NaN,NaN,NaN,NaN
2026-02-11,NaN,1070.3642,NaN,NaN,NaN,NaN,NaN,NaN
2026-02-18,NaN,1071.9487,NaN,NaN,NaN,NaN,NaN,NaN


## 3 · Lending Club Loan Data

> . [Download  dataset from Kaggle -All Lending Club loan data](https://www.kaggle.com/datasets/wordsforthewise/lending-club?select=accepted_2007_to_2018Q4.csv.gz) then use `accepted_2007_to_2018Q4.csv`

Lending Club loans
This is a dataset from Kaggle  with millions of real loans — and most importantly, it tells who paid back their loan and who didn't. This is the training data for the AI model.
This dataset has a collection of  things like:

* Person's credit score
* Their income
* How much debt they already have
* Did they default (not pay back)?


In [13]:
#load dataset

KEY_COLS = [
    'loan_amnt','funded_amnt','term','int_rate','installment',
    'grade','sub_grade','emp_length','home_ownership','annual_inc',
    'verification_status','loan_status','purpose','dti',
    'delinq_2yrs','fico_range_low','fico_range_high',
    'open_acc','pub_rec','revol_bal','revol_util',
    'total_acc','mort_acc','pub_rec_bankruptcies','addr_state','issue_d'
]

DEFAULT_STATUSES = {'Charged Off','Default',
                    'Does not meet the credit policy. Status:Charged Off'}
REPAID_STATUSES  = {'Fully Paid',
                    'Does not meet the credit policy. Status:Fully Paid'}

loans = pd.read_csv('/content/accepted_2007_to_2018Q4 2.csv',
                    usecols=lambda c: c in KEY_COLS, low_memory=False)

loans['default'] = loans['loan_status'].map(
    {**{s:1 for s in DEFAULT_STATUSES}, **{s:0 for s in REPAID_STATUSES}})
loans = loans.dropna(subset=['default'])
loans['default'] = loans['default'].astype(int)

for col in ['int_rate','revol_util']:
    if col in loans.columns:
        loans[col] = pd.to_numeric(loans[col].astype(str).str.replace('%',''), errors='coerce')

loans['issue_date'] = pd.to_datetime(loans['issue_d'], format='%b-%Y', errors='coerce')
loans['issue_year'] = loans['issue_date'].dt.year

loans.to_parquet(f'{BASE}/data/raw/lending_club_raw.parquet', index=False)
print(f'Saved {len(loans):,} loans | {loans["default"].mean()*100:.1f}% default rate')
loans.head()
loans.head()

Saved 1,348,099 loans | 20.0% default rate


,loan_amnt,funded_amnt,term,int_rate,installment,grade,sub_grade,emp_length,home_ownership,annual_inc,...,open_acc,pub_rec,revol_bal,revol_util,total_acc,mort_acc,pub_rec_bankruptcies,default,issue_date,issue_year
0,3600.0,3600.0,36 months,13.99,123.03,C,C4,10+ years,MORTGAGE,55000.0,...,7.0,0.0,2765.0,29.7,13.0,1.0,0.0,0,2015-12-01,2015
1,24700.0,24700.0,36 months,11.99,820.28,C,C1,10+ years,MORTGAGE,65000.0,...,22.0,0.0,21470.0,19.2,38.0,4.0,0.0,0,2015-12-01,2015
2,20000.0,20000.0,60 months,10.78,432.66,B,B4,10+ years,MORTGAGE,63000.0,...,6.0,0.0,7869.0,56.2,18.0,5.0,0.0,0,2015-12-01,2015
4,10400.0,10400.0,60 months,22.45,289.91,F,F1,3 years,MORTGAGE,104433.0,...,12.0,0.0,21929.0,64.5,35.0,6.0,0.0,0,2015-12-01,2015
5,11950.0,11950.0,36 months,13.44,405.18,C,C3,4 years,RENT,34000.0,...,5.0,0.0,8822.0,68.4,6.0,0.0,0.0,0,2015-12-01,2015


# **EDA & Feature Engineering**